In [1]:
# import tensorflow as tf
# # import tensorflow.compat.v1 as tf
# # tf.disable_v2_behavior()
# import tensorflow.compat.v1 as tf
# tf.disable_v2_behavior()
# # from tensorflow.examples.tutorials.mnist import input_data
# # mnist = input_data.read_data_sets('MNIST_data', one_hot=True)
# from tensorflow.keras.datasets import mnist as keras_mnist
# from tensorflow.keras.utils import to_categorical
# import numpy as np

# (x_train, y_train), (x_test, y_test) = keras_mnist.load_data()
# x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
# x_test  = x_test.reshape(-1, 784).astype('float32') / 255.0
# y_train = to_categorical(y_train, 10)
# y_test  = to_categorical(y_test, 10)

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from tensorflow.keras.datasets import mnist as keras_mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

(x_train, y_train), (x_test, y_test) = keras_mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test  = x_test.reshape(-1, 784).astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

class SimpleMNIST:
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels
        self.num_examples = images.shape[0]
        self._idx = 0
    def next_batch(self, batch_size):
        if self._idx + batch_size > self.num_examples:
            perm = np.arange(self.num_examples)
            np.random.shuffle(perm)
            self.images = self.images[perm]
            self.labels = self.labels[perm]
            self._idx = 0
        start = self._idx
        self._idx += batch_size
        return self.images[start:self._idx], self.labels[start:self._idx]

mnist = type('', (), {})()  # empty object
mnist.train = SimpleMNIST(x_train, y_train)
mnist.test = SimpleMNIST(x_test, y_test)

learning_rate = 1e-4
keep_prob_rate = 0.7 # 
max_epoch = 2000

def compute_accuracy(v_xs, v_ys):
    global prediction, sess
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1.0})
    import numpy as np
    pred_labels = np.argmax(y_pre, axis=1)
    true_labels = np.argmax(v_ys, axis=1)
    return np.mean(pred_labels == true_labels)

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # 每一维度  滑动步长全部是 1， padding 方式 选择 same
    # 提示 使用函数  tf.nn.conv2d
    
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')
 

def max_pool_2x2(x):
    # 滑动步长 是 2步; 池化窗口的尺度 高和宽度都是2; padding 方式 请选择 same
    # 提示 使用函数  tf.nn.max_pool
    
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')
 

# define placeholder for inputs to network
xs = tf.placeholder(tf.float32, [None, 784])    # removed /255. because data already normalized
ys = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

#  卷积层 1
## conv1 layer ##

W_conv1 = weight_variable([5, 5, 1, 32])      # patch 5x5, in size 1, out size 32
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1) # 卷积  自己选择 选择激活函数
h_pool1 = max_pool_2x2(h_conv1)              # 池化                           

W_conv2 = weight_variable([5, 5, 32, 64])    # patch 5x5, in size 32, out size 64
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2) # 卷积  自己选择 选择激活函数
h_pool2 = max_pool_2x2(h_conv2)              # 池化


#  全连接层 1
## fc1 layer ##
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])

h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob)

# 全连接层 2
## fc2 layer ##
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)


# 交叉熵函数
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction),
                                              reduction_indices=[1]))
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.Session() as sess:
    init = tf.global_variables_initializer()
    sess.run(init)
    
    for i in range(max_epoch):
        batch_xs, batch_ys = mnist.train.next_batch(100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob:keep_prob_rate})
        if i % 100 == 0:
            print(compute_accuracy(
                mnist.test.images[:1000], mnist.test.labels[:1000]))



Instructions for updating:
non-resource variables are not supported in the long term

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.

0.078
0.834
0.885
0.926
0.941
0.952
0.951
0.962
0.961
0.967
0.971
0.973
0.976
0.978
0.975
0.975
0.975
0.979
0.977
0.979
